# Démo : Reformulation des questions

**Contexte :** assistant du service client de Lagune Banque.

Les questions des clients ne reprennent pas toujours les termes des documents. Cette démo compare trois méthodes de reformulation avant la recherche : réécriture, multi-requête avec fusion RRF et HyDE. Les résultats sont évalués sur des questions courantes et difficiles.

**Sommaire**
- [Configuration](#configuration)
- [Index et questions](#index-questions)
- [Recherche initiale](#recherche-initiale)
- [Réécriture](#reecriture)
- [Multi-requête](#multi-requete)
- [HyDE](#hyde)
- [Évaluation](#evaluation)
- [Synthèse](#synthese)

```mermaid
flowchart LR
    Q["Question du client"] --> R["Réécriture : ra.rewrite"]
    Q --> M["Plusieurs requêtes : ra.variants"]
    M --> F["Fusion RRF : rrf"]
    Q --> H["HyDE : ra.hypothesis"]
    R --> S["Recherche"]
    F --> S
    H --> S
```

Ce notebook ajoute la deuxième capacité de l'assistant : la reformulation.

## <a id="configuration"></a>Configuration

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../starter/code_partage")] if (p / "rag.py").exists()).resolve()))
import rag
import rag_avance as ra
from config import prepare
from weaviate_store import connect

In [ ]:
prepare(embedding=True)

## <a id="index-questions"></a>Index et questions

L'index est celui retenu au TP1 : splitter récursif, 500 caractères, recouvrement 75. Les questions viennent de `data/evaluation/questions_rag.json` :

- 12 **cas courants** : des questions bien formulées ;
- 35 **cas difficiles** : fautes, anglais, nombres exacts, mots seuls, suites de conversation, et 15 questions générées par un modèle avec des mots éloignés de ceux des documents, que la recherche de base rate.

In [ ]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
index = rag.index_chunks(connect(), "index_reformuler", rag.split_documents(documents, 500, 75, "reformuler"))

courants = ra.load_questions("representatif")
difficiles = ra.load_questions("difficile")
pd.Series([x["famille"] for x in difficiles]).value_counts().to_frame("cas difficiles")

## <a id="recherche-initiale"></a>Recherche initiale

La fonction `display_table` affiche les trois premiers chunks pour une requête. Une question de suite, sans son contexte, ne veut rien dire pour la recherche.

In [ ]:
def retrieve(requete, n=10):
    return rag.search(index, requete, n, rag.AUJOURDHUI)


def display_table(requete, n=3):
    for rang, p in enumerate(retrieve(requete, n), 1):
        print(f"[{rang}] {p['meta']['source']} | {p['texte'][:100]!r}")


ECHANGE = "Client : Quels sont les frais de retard du Prêt Express ?\nAssistant : 2 000 FCFA, en forfait."
display_table("Et pour l'Express+ ?")

La recherche ne voit pas l'échange précédent. Le chunk qui répond (les frais de retard de l'Express+, dans la grille tarifaire) n'arrive qu'en troisième position, derrière la fiche produit et une règle de plafond.

## <a id="reecriture"></a>Réécriture

Un modèle de langage réécrit la question pour qu'elle se comprenne seule. `ra.rewrite` reçoit la question et l'échange précédent. Par défaut elle reste fidèle : elle corrige les fautes et résout les renvois, sans rien ajouter.

In [ ]:
requete = ra.rewrite("Et pour l'Express+ ?", ECHANGE)
print(requete)
display_table(requete)

In [ ]:
for question in ["fré de retar prèt expres ??", "c koi le plafon du pret express stp", "Quel est le montant minimum d'un Prêt Express ?"]:
    print(f"{question!r}  ->  {ra.rewrite(question)!r}")

Avec la requête réécrite, le chunk sur les frais de retard passe en première position. Les fautes sont corrigées, et la question déjà claire n'est pas modifiée. Avec `mots_cles=True`, la réécriture enrichit au contraire la requête de synonymes et de termes de métier :

In [ ]:
for question in ["Quels sont les frais de retard du Prêt Express ?", "Combien de temps avant de pouvoir réemprunter ?"]:
    print(f"{question!r}  ->  {ra.rewrite(question, mots_cles=True)!r}")

Ces mots peuvent combler l'écart de vocabulaire avec la documentation, mais ils dégradent une question qui était bonne. La mesure plus bas départage les deux versions.

## <a id="multi-requete"></a>Multi-requête et fusion

La multi-requête demande au modèle trois reformulations, cherche avec chacune, puis fusionne les listes. La fusion **RRF** (Reciprocal Rank Fusion) donne à chaque chunk `1 / (60 + rang)` dans chaque liste et additionne : un chunk bien classé dans plusieurs listes remonte, sans qu'il faille comparer des distances de natures différentes.

`rrf` additionne ces scores par chunk (un chunk est identifié par son texte) et garde les `n` meilleurs.

In [ ]:
def rrf(listes, k=60, n=10):
    scores, vus = {}, {}
    for liste in listes:
        for rang, p in enumerate(liste, 1):
            scores[p["texte"]] = scores.get(p["texte"], 0) + 1 / (k + rang)
            vus[p["texte"]] = p
    return [vus[t] for t, _ in sorted(scores.items(), key=lambda x: -x[1])[:n]]

In [ ]:
question = "Quelle somme maximale puis-je obtenir au total ?"
listes = {"question d'origine": retrieve(question)}
for i, variante in enumerate(ra.variants(question), 1):
    listes[f"variante {i} : {variante[:55]}"] = retrieve(variante)
listes["fusion RRF"] = rrf(list(listes.values()))

pd.DataFrame({nom: [f"{p['meta']['source']} : {p['texte'][:38]!r}" for p in chunks[:3]] for nom, chunks in listes.items()}, index=["1er", "2e", "3e"]).T

## <a id="hyde"></a>HyDE

Une réponse ressemble plus au chunk cherché qu'une question. HyDE demande au modèle d'écrire une réponse plausible, même inventée, et cherche avec elle.

In [ ]:
reponse = ra.hypothesis("Combien de temps avant de pouvoir réemprunter ?")
print(reponse)
display_table(reponse)

## <a id="evaluation"></a>Évaluation

Chaque méthode est appliquée aux 47 questions. Pour chacune, on cherche le rang du premier chunk qui contient la réponse complète. Le **MRR** est la moyenne de `1 / rang` : 1,00 si la bonne réponse est toujours en première position. Les secondes et les appels au modèle sont le coût de la technique.

La mesure dure quelques minutes : chaque technique appelle le modèle de langage pour chaque question.

In [ ]:
METHODES = {
    "vectoriel seul": lambda q, h: retrieve(q),
    "réécriture fidèle": lambda q, h: retrieve(ra.rewrite(q, h)),
    "réécriture avec mots-clés": lambda q, h: retrieve(ra.rewrite(q, h, mots_cles=True)),
    "multi-requête (RRF)": lambda q, h: rrf([retrieve(v) for v in [q] + ra.variants(q)]),
    "HyDE": lambda q, h: retrieve(ra.hypothesis(q)),
}


def mrr(rangs):
    return round(sum(1 / r for r in rangs if r) / len(rangs), 2)


resultats = {}
for nom, methode in METHODES.items():
    resultats[nom] = {"courants": ra.measure(methode, courants), "difficiles": ra.measure(methode, difficiles)}
    print("fait :", nom)

In [ ]:
def subset(rangs, famille):
    return [r for r, x in zip(rangs, difficiles) if x["famille"].startswith(famille)]


pd.DataFrame({
    nom: {
        "cas courants": mrr(r["courants"]["rangs"]),
        "cas difficiles": mrr(r["difficiles"]["rangs"]),
        "dont suites de conversation": mrr(subset(r["difficiles"]["rangs"], "suite")),
        "dont échecs du départ": mrr(subset(r["difficiles"]["rangs"], "générée")),
        "secondes par question": round((r["courants"]["secondes"] * 12 + r["difficiles"]["secondes"] * 35) / 47, 1),
        "appels au modèle": round((r["courants"]["appels LLM"] * 12 + r["difficiles"]["appels LLM"] * 35) / 47, 1),
    } for nom, r in resultats.items()}).T

## <a id="synthese"></a>Synthèse

1 est le MRR maximal, et le vectoriel seul est la référence de chaque colonne. Les chiffres varient légèrement d'une exécution à l'autre ; l'ordre de grandeur reste le même.

- **La réécriture fidèle règle les suites de conversation** (de 0,7 à 1,0 environ) sans toucher aux cas courants. C'est la plus sûre.
- **La réécriture avec mots-clés aide les questions dont le vocabulaire diffère** des documents (les échecs du départ, de 0,30 à 0,4 environ), mais **dégrade les cas courants** (de 0,80 à 0,7 environ) : les mots ajoutés diluent une question qui était bonne.
- **La multi-requête et HyDE gagnent peu** sur les échecs du départ (jusqu'à 0,37 et 0,38) pour deux à quatre fois plus de temps. HyDE ne voit pas l'échange précédent : il est moins bon que le vectoriel seul sur les suites de conversation.
- **Aucune technique ne règle les échecs du départ** : ils restent sous 0,45. La reformulation ne suffit pas, il faut agir sur le classement des chunks.

| Technique | Aide quand | Nuit quand | Coût |
|---|---|---|---|
| Réécriture fidèle | la question dépend de l'échange précédent, ou contient des fautes | presque jamais | un appel au modèle |
| Réécriture avec mots-clés | le client n'emploie pas les mots de la documentation | la question était déjà claire | un appel au modèle |
| Multi-requête (RRF) | la formulation s'écarte de la documentation | le gain est faible face au temps ajouté | un appel au modèle et plusieurs recherches : environ 4 fois plus long |
| HyDE | le vocabulaire s'écarte de la documentation | la question dépend de l'échange précédent | un appel au modèle |

Aucune technique ne gagne partout, et aucune ne règle les cas les plus difficiles : on les mesure avant de les retenir. Le TP2 ajoute les deux techniques qui ne touchent pas à la question : la recherche par mots-clés (BM25) et le reranking.